## Contraindication IE + Normalization: VO

### 2026/04/23

### VO Statistics: 
1. Number of VO classes mapped to RxNorm : 2060
2. Number of SPL documents associated : ~~279~~ 88

#### Loading VO

In [1]:
%cd /data/wmanuel3/VaxMapperRepo/
!pwd

/data/wmanuel3/VaxMapperRepo
/data/wmanuel3/VaxMapperRepo


In [2]:
from owlready2 import * 
from owlready2 import IRIS
import pandas as pd
import requests
import pickle
import ast


In [3]:
onto = get_ontology("vo_source/VO_2026-03-06.owl").load()
res = []
for cls in onto.classes():
    if cls.VO_0003198: #RxNorm Annotation
        id = cls.name
        label = cls.label[0] if cls.label else None
        rx = cls.VO_0003198[0] if cls.VO_0003198 else None
        res.append((id, label, rx))
print(f"Number of classes with RxNorm annotation: {len(res)}")

Number of classes with RxNorm annotation: 2060


#### Extract and normalize contraindications of SPLs related to RxNorm annotations: 

In [4]:
df = pd.DataFrame(res, columns=["Class ID", "Class Label", "RxNorm Code"])
print(df.head())

     Class ID                                        Class Label RxNorm Code
0  VO_0003384                tetanus toxoid vaccine, inactivated      798306
1  VO_0003455             diphtheria toxoid vaccine, inactivated      798304
2  VO_0004208      BACILLUS ANTHRACIS STRAIN V770-NP1-R ANTIGENS     1368371
3  VO_0003150                Hepatitis B surface antigen vaccine      797752
4  VO_0003408  Haemophilus influenzae b (Ross strain) capsula...      798444


##### Helper functions: 

In [5]:
# API request for RxNorm
def get_rx_prop_v2(rxnorm_code, property_name="splSetIdItem"):
    base_url = f"https://rxnav.nlm.nih.gov"
    url_1 = f"{base_url}/REST/ndcproperties.json"
    params = {"id": rxnorm_code}
    response = requests.get(url_1, params=params)
    res = response.json()
    # resu = res.get("propConceptGroup", {}).get("propConcept", [])[0]
    result = [r.get(property_name) for r in res.get("ndcPropertyList", {}).get("ndcProperty", []) if r.get(property_name)]
    return result
    # return res

In [6]:
def get_open_fda_prop(term=None, property_name="openfda.product_ndc", section="contraindications"):
    base_url = f"https://api.fda.gov/drug/label.json"
    params = {"search": f"{property_name}:{term}"}
    response = requests.get(base_url, params=params)
    res = response.json().get("results", [])
    result = [r.get(section, []) for r in res]
    return result 
    # return res

In [21]:
from src.extraction.section_parser import CONTRA_Loinc, extract_section

In [46]:
def extract_section_safe(spl_set_id, loinc_codes):
    try:
        return extract_section(str(spl_set_id), loinc_codes)['sections'][loinc_codes[0]]['section_text']
    except Exception as e:
        print(f"Error extracting section for SPL Set ID {spl_set_id}: {e}")
        return "N/A"

In [51]:
import requests
from bs4 import BeautifulSoup

def extract_section_from_page(spl_set_id, loinc_codes):
    """Fallback: scrape DailyMed page source for archived SPL IDs."""
    url = f"https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid={spl_set_id}"
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for loinc_code in loinc_codes:
        div = soup.find("div", {"class": "Section", "data-sectioncode": loinc_code})
        if div:
            return div.get_text(separator=" ", strip=True)

    return None


def extract_section_safe(spl_set_id, loinc_codes):
    # Primary method
    try:
        return extract_section(str(spl_set_id), loinc_codes)['sections'][loinc_codes[0]]['section_text']
    except Exception as primary_err:
        print(f"Primary extraction failed for {spl_set_id}: {primary_err}. Trying page scrape...")

    # Fallback: scrape page source
    try:
        text = extract_section_from_page(spl_set_id, loinc_codes)
        if text:
            return text
        print(f"Section not found in page source for {spl_set_id}")
    except Exception as fallback_err:
        print(f"Page scrape also failed for {spl_set_id}: {fallback_err}")

    return "N/A"

In [52]:
extract_section_safe("276b84fb-096b-3f91-e054-00144ff8d46c", [CONTRA_Loinc])

Primary extraction failed for 276b84fb-096b-3f91-e054-00144ff8d46c: '34070-3'. Trying page scrape...
Section not found in page source for 276b84fb-096b-3f91-e054-00144ff8d46c


'N/A'

In [49]:
extract_section("276b84fb-096b-3f91-e054-00144ff8d46c", [CONTRA_Loinc])

{'setid': '276b84fb-096b-3f91-e054-00144ff8d46c',
 'product_name': 'Ixiaro',
 'found': False,
 'sections': {}}

##### Contraindication lookup: 

In [9]:
df['spl_set_id'] = df['RxNorm Code'].apply(lambda x: get_rx_prop_v2(x, "splSetIdItem"))

In [ ]:
contra_df = df[df['spl_set_id'].str.len() > 0]
contra_df = contra_df.explode('spl_set_id')

,Class ID,Class Label,RxNorm Code,spl_set_id
89,VO_0019370,1 ML hepatitis B surface antigen vaccine 0.04 ...,830263,b5bd99e4-569d-48d5-ba75-16e69f8c409a
89,VO_0019370,1 ML hepatitis B surface antigen vaccine 0.04 ...,830263,b5bd99e4-569d-48d5-ba75-16e69f8c409a
97,VO_0003238,0.5 ML Hepatitis B Surface Antigen Vaccine 0.0...,1658155,b5bd99e4-569d-48d5-ba75-16e69f8c409a
97,VO_0003238,0.5 ML Hepatitis B Surface Antigen Vaccine 0.0...,1658155,b5bd99e4-569d-48d5-ba75-16e69f8c409a
97,VO_0003238,0.5 ML Hepatitis B Surface Antigen Vaccine 0.0...,1658155,f1ad4bca-839d-41cd-a132-a6984780912e
...,...,...,...,...
1856,VO_0020210,"SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 0...",2623382,d19d59cb-f1cd-479d-ab0c-a36971e65544
1860,VO_0020214,"SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 0...",2621900,d19d59cb-f1cd-479d-ab0c-a36971e65544
1860,VO_0020214,"SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 0...",2621900,d19d59cb-f1cd-479d-ab0c-a36971e65544
1862,VO_0020216,"SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 0...",2610319,d19d59cb-f1cd-479d-ab0c-a36971e65544


In [55]:
lookup_df = contra_df.groupby('spl_set_id')['Class ID'].apply(list).reset_index()
lookup_df['contraindications'] = lookup_df['spl_set_id'].apply(lambda x: extract_section_safe(str(x), [CONTRA_Loinc]))

Primary extraction failed for 01dae2ac-8332-4e23-b971-67dc36c3ee16: Failed to fetch SPL XML for setid=01dae2ac-8332-4e23-b971-67dc36c3ee16: 404. Trying page scrape...
Primary extraction failed for 04c2d0f3-6209-4ae8-823c-842d0881b61b: Failed to fetch SPL XML for setid=04c2d0f3-6209-4ae8-823c-842d0881b61b: 404. Trying page scrape...
Primary extraction failed for 168a670b-0cbb-067c-e054-00144ff88e88: Failed to fetch SPL XML for setid=168a670b-0cbb-067c-e054-00144ff88e88: 404. Trying page scrape...
Primary extraction failed for 276b84fb-096b-3f91-e054-00144ff8d46c: '34070-3'. Trying page scrape...
Section not found in page source for 276b84fb-096b-3f91-e054-00144ff8d46c
Primary extraction failed for 377116cf-adfe-40b5-b871-3a0fc8b4103e: Failed to fetch SPL XML for setid=377116cf-adfe-40b5-b871-3a0fc8b4103e: 404. Trying page scrape...
Primary extraction failed for 3aa6e7be-0f88-4a16-83ed-023b510ed6a5: Failed to fetch SPL XML for setid=3aa6e7be-0f88-4a16-83ed-023b510ed6a5: 404. Trying page 

In [61]:
vo_spl = lookup_df['spl_set_id'].tolist()
with open("vo_source/vo_spl_ids.txt", "w") as f:
    for spl_id in vo_spl:
        f.write(f"{spl_id}\n")

In [56]:
lookup_df.to_csv("vo_source/vo_spl_contraindications.csv", index=False)

In [53]:
df.to_pickle("vo_source/VO_20260306_Rx_SPL_V2.pkl")
df

,Class ID,Class Label,RxNorm Code,spl
0,VO_0003384,"tetanus toxoid vaccine, inactivated",798306,[]
1,VO_0003455,"diphtheria toxoid vaccine, inactivated",798304,[]
2,VO_0004208,BACILLUS ANTHRACIS STRAIN V770-NP1-R ANTIGENS,1368371,[]
3,VO_0003150,Hepatitis B surface antigen vaccine,797752,[]
4,VO_0003408,Haemophilus influenzae b (Ross strain) capsula...,798444,[]
...,...,...,...,...
2055,VO_0021174,"0.3 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT...",2664846,[]
2056,VO_0021176,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664819,[]
2057,VO_0021177,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664811,[]
2058,VO_0021178,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664822,[]


### Lookup VO ID by SPL_ID

In [71]:
reverse_lookup_agg = (
    df[df['spl'].map(len) > 0]
    .explode('spl')
    .groupby('spl')[['Class ID', 'Class Label', 'RxNorm Code']]
    .agg(list)
    .to_dict(orient='index')
)
reverse_lookup_agg_df = pd.DataFrame.from_dict(reverse_lookup_agg , orient='index').reset_index()
reverse_lookup_agg_df = reverse_lookup_agg_df.rename(columns={'index': 'spl'})
reverse_lookup_agg_df = reverse_lookup_agg_df.explode(list(reverse_lookup_agg_df.columns.difference(['spl'])))
reverse_lookup_agg_df

,spl,Class ID,Class Label,RxNorm Code
0,,VO_0003264,1 ML Hepatitis B Surface Antigen Vaccine 0.02 ...,798428
0,,VO_0003356,Typhoid Vaccine Live Ty21a 2000000000 UNT Dela...,762602
0,,VO_0003356,Typhoid Vaccine Live Ty21a 2000000000 UNT Dela...,762602
0,,VO_0003378,"0.5 ML Yellow-Fever Virus Vaccine, 17D-204 str...",1876710
0,,VO_0003831,"Havrix, Hepatitis A Vaccine (Inactivated), (Pe...",798482
...,...,...,...,...
86,f9499a4d-1288-4bd3-9d59-1d72092c38cd,VO_0003832,"Havrix, Hepatitis A Vaccine (Inactivated), (Ad...",798479
86,f9499a4d-1288-4bd3-9d59-1d72092c38cd,VO_0003832,"Havrix, Hepatitis A Vaccine (Inactivated), (Ad...",798479
87,fb8f92de-56eb-41e3-b3f8-f1b9911e3c40,VO_0019737,influenza A virus A/Darwin/6/2021 (H3N2) antig...,2605561
87,fb8f92de-56eb-41e3-b3f8-f1b9911e3c40,VO_0019737,influenza A virus A/Darwin/6/2021 (H3N2) antig...,2605561


In [25]:
reverse_lookup_agg_df[reverse_lookup_agg_df['spl'] == '5a56ddc9-2550-4a6a-a553-3bc919b844f0']

,spl,Class ID,Class Label,RxNorm Code
93,5a56ddc9-2550-4a6a-a553-3bc919b844f0,VO_0003384,"tetanus toxoid vaccine, inactivated",798306


In [54]:
reverse_lookup = df[df['spl'].map(len) > 0].explode('spl').set_index('spl')['RxNorm Code'].to_dict()

In [55]:
reverse_lookup_df = pd.DataFrame(list(reverse_lookup.items()), columns=['SPL_SET_ID', 'RxNorm Code'])
print(reverse_lookup_df.head())

                             SPL_SET_ID RxNorm Code
0  b5bd99e4-569d-48d5-ba75-16e69f8c409a      830253
1  f1ad4bca-839d-41cd-a132-a6984780912e     1658155
2  e80cfadc-71b6-4881-b5f7-2be67adbf8b8     1658150
3  dcecf9e0-6c28-4964-9ddb-9379705fa26c      798428
4  2ec65f7e-4aa2-4b41-b578-885ea59d6e9d      798430


In [72]:
reverse_lookup_df

,SPL_SET_ID,RxNorm Code
0,b5bd99e4-569d-48d5-ba75-16e69f8c409a,830253
1,f1ad4bca-839d-41cd-a132-a6984780912e,1658155
2,e80cfadc-71b6-4881-b5f7-2be67adbf8b8,1658150
3,dcecf9e0-6c28-4964-9ddb-9379705fa26c,798428
4,2ec65f7e-4aa2-4b41-b578-885ea59d6e9d,798430
...,...,...
84,3aa6e7be-0f88-4a16-83ed-023b510ed6a5,2605739
85,fb8f92de-56eb-41e3-b3f8-f1b9911e3c40,2605561
86,04c2d0f3-6209-4ae8-823c-842d0881b61b,2623378
87,dc9e86ab-f023-40f8-9b61-d9039bdcea1d,2610328


In [56]:
print(f"Unique SPL_SET_IDs: {len(reverse_lookup_df['SPL_SET_ID'].unique())}")

Unique SPL_SET_IDs: 89


In [57]:
VO_SPL = (reverse_lookup_df['SPL_SET_ID'].unique().tolist())

In [58]:
with open("results/VO_SPL_V2.txt", "w") as f:
    for spl in VO_SPL:
        f.write(f"{spl}\n")

### Baseline SPL Run: 

In [4]:
from src.extraction.section_parser import CONTRA_Loinc, extract_section

### Extraction Error Analysis 

In [60]:
results_csv = "results/20260415_isolation/011_anc_strict/evaluation_details.csv"
extraction_cache = "results/20260415_isolation/extraction_cache.jsonl"

In [69]:
import json
import csv

# Find how many of the n=100 SPL_SET_IDs in the evaluation was incorrectly extracted @ 0.8 similarity threshold, and what those SPL_SET_IDs are.
with open(results_csv, "r") as f:
    reader = csv.DictReader(f)
    incorrect_spls = [row for row in reader if row['contraindication_tp'] == '0']

In [70]:
[(r['annotation'], r['query_text']) for r in incorrect_spls]

[('viral diseases of the cornea',
  'viral diseases of the cornea and conjunctiva'),
 ('viral diseases of the conjunctiva',
  'varicella of the cornea and conjunctiva'),
 ('Varicella', ''),
 ('Vaccinia', 'vaccinia of the cornea and conjunctiva'),
 ('Mycobacterial infection of the eye', 'Mycobacterial infection of the eye'),
 ('Hypersensitivity to any component of the medication',
  'Hypersensitivity to a component of the medication'),
 ('Fungal diseases of ocular structures',
  'Fungal diseases of ocular structures'),
 ('Epithelial herpes simplex keratitis',
  'Epithelial herpes simplex keratitis (dendritic keratitis)'),
 ('Asthma after taking aspirin', 'asthma after taking aspirin'),
 ('Allergy-type reactions after taking NSAIDs',
  'allergic-type reactions after taking other NSAIDs'),
 ('Allergy-type reactions after taking aspirin',
  'allergic-type reactions after taking aspirin'),
 ('Hypersensitivity to bovine protein',
  'known hypersensitivity to bovine protein'),
 ('Hypersensiti